In [ ]:
# ── EXP-01: Encoder-Decoder Binario (baseline) ───────────────────────────────
data_dir    = 'xBD_UC3M'
patch_size  = 128
num_epochs  = 8
batch_size  = 4
num_workers = 0

BASE_RESULT_DIR = r'/Users/juan.macias@feverup.com/Desktop/cv/cv-2a-image-segmentation/VA_Pr2A_ImageSegmentation_2025_2026/results/parte-4'
result_dir      = os.path.join(BASE_RESULT_DIR, 'exp01_binary_encoder_decoder')
model_name      = 'encoder_decoder_binary_exp01'

num_classes = 2
class_names = ['background', 'building']

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Device: {device}')

# ── Dataset statistics (computed once, then cached) ───────────────────────────
stats_cache = os.path.join(BASE_RESULT_DIR, 'xbd_stats.npy')
if os.path.exists(stats_cache):
    image_stats = np.load(stats_cache, allow_pickle=True).item()
    print(f"Stats loaded from cache — mean={image_stats['mean']}  std={image_stats['std']}")
else:
    print('Computing dataset statistics (runs once, then cached) ...')
    _stats_ds   = xBDDataset(data_dir, split=['train'], task='segmentation',
                              patch_size=patch_size, stats=None)
    image_stats = _stats_ds.compute_statistics()
    np.save(stats_cache, image_stats)
    print(f"Stats saved — mean={image_stats['mean']}  std={image_stats['std']}")

# ── Datasets (no augmentation) ────────────────────────────────────────────────
train_ds_raw = xBDDataset(data_dir, split=['train'], task='segmentation',
                           patch_size=patch_size, stats=image_stats, transform=None)
val_ds_raw   = xBDDataset(data_dir, split=['val'],   task='segmentation',
                           patch_size=patch_size, stats=image_stats, transform=None)
test_ds_raw  = xBDDataset(data_dir, split=['test'],  task='segmentation',
                           patch_size=patch_size, stats=image_stats, transform=None)

class KeyAdapterBinary(Dataset):
    def __init__(self, ds):
        self.ds = ds
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, idx):
        s = self.ds[idx]
        mask_binary = (s['mask_patch_raw'] != 255).long()
        return {'image': s['patch_post'],
                'mask':  mask_binary,
                'img_path': s['patch_post_path']}

dataloaders = {
    'Train': DataLoader(KeyAdapterBinary(train_ds_raw), batch_size=batch_size,
                        shuffle=True,  num_workers=num_workers),
    'Val':   DataLoader(KeyAdapterBinary(val_ds_raw),   batch_size=1,
                        shuffle=False, num_workers=num_workers),
    'Test':  DataLoader(KeyAdapterBinary(test_ds_raw),  batch_size=1,
                        shuffle=False, num_workers=num_workers),
}

# ── Simple Encoder-Decoder from scratch ───────────────────────────────────────
# No BatchNorm: train_model calls set_bn_eval() which would freeze BN running
# stats before they accumulate — incompatible with training from scratch.
# Plain Conv→ReLU blocks are stable enough at this scale.
class EncoderDecoder(torch.nn.Module):
    """
    3-level encoder-decoder (no skip connections).
    Spatial flow for patch_size=128:
      Input  (B,  3, 128, 128)
      enc1   (B, 32, 128, 128) → pool → (B, 32, 64, 64)
      enc2   (B, 64,  64,  64) → pool → (B, 64, 32, 32)
      enc3   (B,128,  32,  32) → pool → (B,128, 16, 16)
      btn    (B,256,  16,  16)
      dec3   upsample → (B,128, 32, 32)
      dec2   upsample → (B, 64, 64, 64)
      dec1   upsample → (B, 32,128,128)
      head   (B, num_classes, 128, 128)
    """
    def __init__(self, num_classes):
        super().__init__()
        self.pool = torch.nn.MaxPool2d(2)
        self.enc1 = self._block(3,   32)
        self.enc2 = self._block(32,  64)
        self.enc3 = self._block(64, 128)
        self.btn  = self._block(128, 256)
        self.up   = torch.nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.dec3 = self._block(256, 128)
        self.dec2 = self._block(128,  64)
        self.dec1 = self._block(64,   32)
        self.head = torch.nn.Conv2d(32, num_classes, kernel_size=1)

    @staticmethod
    def _block(in_ch, out_ch):
        return torch.nn.Sequential(
            torch.nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            torch.nn.ReLU(inplace=True),
        )

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b  = self.btn(self.pool(e3))
        d  = self.dec3(self.up(b))
        d  = self.dec2(self.up(d))
        d  = self.dec1(self.up(d))
        return {'out': self.head(d)}  # dict matches train_model's outputs['out']


In [ ]:
model = EncoderDecoder(num_classes).to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

# ── Loss: uniform CrossEntropy (no class weighting) ───────────────────────────
criterion = torch.nn.CrossEntropyLoss(reduction='mean')

# ── Optimiser + StepLR schedule ───────────────────────────────────────────────
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
lr_sched  = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)
metrics   = {'jaccard_score': jaccard_score}

os.makedirs(result_dir, exist_ok=True)

# ── Train ─────────────────────────────────────────────────────────────────────
model = train_model(
    model, criterion, dataloaders, device,
    optimizer, lr_sched, metrics,
    result_dir, model_name,
    num_classes=num_classes, num_epochs=num_epochs,
)

# ── Evaluate on test set ──────────────────────────────────────────────────────
weights = torch.load(os.path.join(result_dir, model_name + '_best.pth.tar'))['state_dict']
model.to(device)
model.load_state_dict(weights)
model.eval()
cm_total = test_segmentation_model(model, dataloaders, num_classes, class_names, result_dir, True, batchsize_test)
